In [ ]:
!pip install scikit-learn
!pip install xgboost
!pip install smote-variants

In [ ]:
import pandas as pd

# =========================================
# LOAD RAW DATASET
# =========================================

df = pd.read_csv(
    '/kaggle/input/datasets/aderahmad/stroke-dataset-raw/healthcare-dataset-stroke-data.csv'
)

# Hapus ID
df = df.drop('id', axis=1)

X = df.drop('stroke', axis=1)
y = df['stroke']

print("Dataset shape :", X.shape)
print("\nClass distribution:")
print(y.value_counts())

In [ ]:
import numpy as np
import pandas as pd
import smote_variants as sv
import time
import sklearn
import xgboost

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
    auc, confusion_matrix, balanced_accuracy_score, matthews_corrcoef
)

SEED = 42

print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("smote-variants:", getattr(sv, "__version__", "version not exposed"))

# =========================================
# FEATURES AND ENCODING
# =========================================
numeric_features = [
    'age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi'
]
onehot_features = ['gender', 'work_type', 'smoking_status']
binary_features = ['ever_married', 'Residence_type']
binary_categories = [['No', 'Yes'], ['Rural', 'Urban']]

# Tree-based models: mean imputation, no scaling.
numeric_transformer_tree = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])
onehot_transformer_tree = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
binary_transformer_tree = Pipeline(steps=[
    ('encoder', OrdinalEncoder(
        categories=binary_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])
preprocessor_tree = ColumnTransformer(transformers=[
    ('num', numeric_transformer_tree, numeric_features),
    ('onehot', onehot_transformer_tree, onehot_features),
    ('binary', binary_transformer_tree, binary_features)
])

# Linear models: same imputation/encoding, with scaling for numeric features.
numeric_transformer_linear = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])
onehot_transformer_linear = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
binary_transformer_linear = Pipeline(steps=[
    ('encoder', OrdinalEncoder(
        categories=binary_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])
preprocessor_linear = ColumnTransformer(transformers=[
    ('num', numeric_transformer_linear, numeric_features),
    ('onehot', onehot_transformer_linear, onehot_features),
    ('binary', binary_transformer_linear, binary_features)
])

# =========================================
# MODELS: MATCH BASELINE CONFIGURATIONS
# =========================================
# DT is added as requested. No model uses class_weight in this experiment.
model_specs = {
    'Logistic Regression': {
        'preprocessor': 'linear',
        'estimator': LogisticRegression(random_state=SEED, max_iter=1000)
    },
    'Random Forest': {
        'preprocessor': 'tree',
        'estimator': RandomForestClassifier(random_state=SEED)
    },
    'XGBoost': {
        'preprocessor': 'tree',
        'estimator': XGBClassifier(random_state=SEED, eval_metric='logloss')
    },
    'SVM': {
        'preprocessor': 'linear',
        'estimator': SVC(probability=True, random_state=SEED)
    }
}

# =========================================
# REPEATED STRATIFIED CROSS-VALIDATION
# =========================================
rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=2, random_state=SEED)
cv_splits = list(rskf.split(X, y))
print(f"Total CV splits per model: {len(cv_splits)}")

# Use smote-variants (not imbalanced-learn).
# Oversampling is performed only on each training fold after preprocessing.
def make_smote():
    return sv.SMOTE(random_state=SEED)

metric_names = [
    'accuracy', 'precision', 'recall', 'specificity', 'f1',
    'roc_auc', 'average_precision', 'pr_auc', 'balanced_accuracy', 'mcc'
]
results = {name: {metric: [] for metric in metric_names} for name in model_specs}
fold_results = []

for split_number, (train_idx, test_idx) in enumerate(cv_splits, start=1):
    repeat = ((split_number - 1) // 10) + 1
    fold = ((split_number - 1) % 10) + 1
    print(f"\n========== Repeat {repeat} - Fold {fold} ==========")

    X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

    before = y_train.value_counts().sort_index()
    print(f"Before SMOTE — majority (0): {before.get(0, 0)}, minority (1): {before.get(1, 0)}")

    # Cache processed/resampled data by preprocessing family, so tree models
    # share the same transformed data and the same SMOTE realization.
    fold_data = {}
    for prep_name, prep in [('tree', preprocessor_tree), ('linear', preprocessor_linear)]:
        X_train_processed = prep.fit_transform(X_train)
        X_test_processed = prep.transform(X_test)

        smote_start = time.perf_counter()
        X_train_resampled, y_train_resampled = make_smote().sample(
            X_train_processed, y_train.to_numpy()
        )
        smote_runtime = time.perf_counter() - smote_start

        after = pd.Series(y_train_resampled).value_counts().sort_index()
        fold_data[prep_name] = {
            'X_train': X_train_resampled,
            'y_train': y_train_resampled,
            'X_test': X_test_processed,
            'smote_runtime': smote_runtime,
            'after': after
        }

    fold_row = {
        'Repeat': repeat,
        'Fold': fold,
        'Before_Majority': int(before.get(0, 0)),
        'Before_Minority': int(before.get(1, 0)),
    }

    for model_name, spec in model_specs.items():
        data = fold_data[spec['preprocessor']]
        model = spec['estimator']

        train_start = time.perf_counter()
        model.fit(data['X_train'], data['y_train'])
        training_runtime = time.perf_counter() - train_start

        y_pred = model.predict(data['X_test'])
        y_prob = model.predict_proba(data['X_test'])[:, 1]

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)

        fold_metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, zero_division=0),
            'recall': recall_score(y_test, y_pred, zero_division=0),
            'specificity': specificity,
            'f1': f1_score(y_test, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_test, y_prob),
            'average_precision': average_precision_score(y_test, y_prob),
            'pr_auc': auc(recall_curve, precision_curve),
            'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
            'mcc': matthews_corrcoef(y_test, y_pred)
        }
        for metric, value in fold_metrics.items():
            results[model_name][metric].append(value)

        prefix = {
            'Logistic Regression': 'LR',
            'Random Forest': 'RF',
            'XGBoost': 'XGB',
            'SVM': 'SVM'
        }[model_name]

        fold_row[f'{prefix}_Training_Runtime_Seconds'] = training_runtime
        fold_row[f'{prefix}_SMOTE_Runtime_Seconds'] = data['smote_runtime']
        for metric, value in fold_metrics.items():
            fold_row[f'{prefix}_{metric}'] = value
        fold_row.update({
            f'{prefix}_TN': int(tn),
            f'{prefix}_FP': int(fp),
            f'{prefix}_FN': int(fn),
            f'{prefix}_TP': int(tp)
        })

    # Both preprocessing families can produce different neighbour geometries,
    # so record their actual post-SMOTE counts separately.
    for prep_name, data in fold_data.items():
        after = data['after']
        fold_row[f'{prep_name}_After_Majority'] = int(after.get(0, 0))
        fold_row[f'{prep_name}_After_Minority'] = int(after.get(1, 0))
        fold_row[f'{prep_name}_SMOTE_Runtime_Seconds'] = data['smote_runtime']

    fold_results.append(fold_row)

print("\nAll folds completed.")


In [ ]:
# =========================================
# SUMMARY PER MODEL
# =========================================
for model_name, model_result in results.items():
    print(f"\n===== {model_name.upper()} =====")
    for metric, values in model_result.items():
        print(f"{metric:20s} Mean: {np.mean(values):.4f} | SD: {np.std(values, ddof=1):.4f}")


In [ ]:
from scipy import stats

# Create fold-level DataFrame first, then use it for all summaries.
fold_df = pd.DataFrame(fold_results)
fold_df.to_csv('smote_fold_results.csv', index=False)

# =========================================
# METRIC SUMMARY + DESCRIPTIVE 95% CI
# =========================================
summary_rows = []
for model_name, model_result in results.items():
    for metric, values in model_result.items():
        values = np.asarray(values, dtype=float)
        mean = float(np.mean(values))
        std = float(np.std(values, ddof=1))
        n = len(values)
        sem = stats.sem(values)
        ci_low, ci_high = stats.t.interval(
            0.95, df=n - 1, loc=mean, scale=sem
        )
        summary_rows.append({
            'Model': model_name,
            'Metric': metric,
            'Mean': mean,
            'Std': std,
            'CI_95_Lower_descriptive': ci_low,
            'CI_95_Upper_descriptive': ci_high,
            'N_fold_results': n
        })

results_df = pd.DataFrame(summary_rows)
results_df.to_csv('smote_results.csv', index=False)
print("Saved smote_results.csv and smote_fold_results.csv")

# Note: repeated-CV fold scores are dependent. These t-based intervals are
# descriptive and should not be presented as dependence-corrected inferential CIs.

# =========================================
# RUNTIME SUMMARY
# =========================================
runtime_rows = []
for model_name, prefix in [
    ('Logistic Regression', 'LR'),
    ('Random Forest', 'RF'),
    ('XGBoost', 'XGB'),
    ('SVM', 'SVM')
]:
    col = f'{prefix}_Training_Runtime_Seconds'
    smote_col = f'{prefix}_SMOTE_Runtime_Seconds'
    runtime_rows.append({
        'Model': model_name,
        'Mean_Training_Runtime_Seconds': fold_df[col].mean(),
        'SD_Training_Runtime_Seconds': fold_df[col].std(),
        'Mean_SMOTE_Runtime_Seconds': fold_df[smote_col].mean(),
        'SD_SMOTE_Runtime_Seconds': fold_df[smote_col].std()
    })

runtime_summary = pd.DataFrame(runtime_rows)
runtime_summary.to_csv('smote_runtime.csv', index=False)
print(runtime_summary.to_string(index=False))
print("Saved smote_runtime.csv")
